# Benchmark Comparativo de SLMs sobre el Pipeline V2

Evalúa comparativamente 5 modelos de lenguaje pequeños (SLMs) sobre el pipeline completo V2
del chatbot de apoyo emocional para ciberacoso adolescente.

**Modelos evaluados:** TinyLlama · Gemma 2B · Phi-3 Mini · Mistral 7B · Gemma 7B

**Nota sobre el RAG:** La calidad del sistema de recuperación ya fue evaluada en los Experimentos 1-4.
Aquí el RAG está fijo para todos los modelos; lo que varía es cómo cada SLM utiliza el contexto
recuperado, lo que queda capturado especialmente en DIM 2.

## Hipótesis

- **H1 (Eficiencia):** Los modelos pequeños (TinyLlama, Gemma 2B) tendrán latencias significativamente
  menores, pero a costa de calidad clínica. Se espera una relación inversa entre tamaño y eficiencia.
- **H2 (Calidad clínica):** Los modelos de 7B parámetros (Mistral, Gemma 7B) producirán respuestas
  terapéuticamente más adecuadas gracias a su mayor capacidad de razonamiento contextual y una
  integración más precisa del contexto RAG.
- **H3 (Coherencia):** Los modelos más potentes mantendrán mejor el hilo conversacional sin
  preguntar información redundante ya aportada por el usuario.

## Metodología

### Warmup — primera inferencia y semilla

La primera inferencia de cada modelo incluye el tiempo de carga en VRAM, lo que infla artificialmente
la latencia real. Para cada modelo se ejecuta **una inferencia de calentamiento** con el mensaje `"hola"`
antes de medir. Solo a partir de la segunda inferencia se registran las métricas.

Para garantizar la reproducibilidad del benchmark, `ChatOllama` se instancia con el parámetro
`seed=112` en todas las llamadas de medición.

### Generación única de respuestas

Las respuestas se generan **una sola vez** por modelo para los 15 casos y se almacenan en
`results[model][case_id]`. Las dimensiones DIM 1 y DIM 2 reutilizan este diccionario sin regenerar.
DIM 3 genera aparte las conversaciones de 3 turnos.

### Persistencia parcial

Tras completar cada modelo se guardan los resultados en `results/slm_benchmark_results.json` para
evitar pérdida de datos si el kernel falla.

## 0. Imports y configuración

In [ ]:
import json
import re
import subprocess
import sys
import time
import unicodedata
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from langchain_ollama import ChatOllama
from tqdm.notebook import tqdm


def save_plot(filename: str, dpi: int = 300) -> None:
    """Guarda la figura activa en docs/figures con alta resolución."""
    output_dir = Path('../docs/figures')
    output_dir.mkdir(parents=True, exist_ok=True)
    if not filename.endswith('.png'):
        filename += '.png'
    plt.savefig(output_dir / filename, bbox_inches='tight', dpi=dpi)
    print(f"Gráfica guardada: {filename}")


# Estilo global
sns.set_theme(style='whitegrid', palette='muted', font_scale=1.15)
plt.rcParams.update({'figure.dpi': 130, 'figure.facecolor': 'white'})

PROJECT_ROOT = Path('..').resolve()
sys.path.insert(0, str(PROJECT_ROOT))

RESULTS_PATH = PROJECT_ROOT / 'eval' / 'slm_results' / 'slm_benchmark_results.json'
FIGURES_DIR  = PROJECT_ROOT / 'docs' / 'figures'
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

# Modelos a evaluar
MODELS = ["tinyllama", "gemma:2b", "phi3:mini", "mistral:7b", "gemma:7b"]

# VRAM por modelo
VRAM_GB = {
    "tinyllama": 0.6,
    "gemma:2b":  1.7,
    "phi3:mini": 2.2,
    "mistral:7b": 4.4,
    "gemma:7b":  5.0,
}

## 1. Check de disponibilidad de modelos Ollama

In [ ]:
def get_ollama_models() -> list[str]:
    """Devuelve la lista de modelos disponibles en el servidor Ollama local."""
    try:
        result = subprocess.run(
            ['ollama', 'list'],
            capture_output=True, text=True, timeout=15
        )
        lines = result.stdout.strip().splitlines()
        # La primera línea es la cabecera; el nombre del modelo es la primera columna
        return [line.split()[0] for line in lines[1:] if line.strip()]
    except Exception as e:
        print(f"[ADVERTENCIA] No se pudo consultar Ollama: {e}")
        return []


available = get_ollama_models()
print(f"Modelos disponibles en Ollama: {available}\n")

missing = []
for m in MODELS:
    # Comprobación flexible: el nombre puede incluir etiqueta ':latest'
    if not any(m in av for av in available):
        missing.append(m)
        print(f"[ADVERTENCIA] Modelo '{m}' NO encontrado en Ollama. "
              f"Ejecuta: ollama pull {m}")
    else:
        print(f"[OK] {m}")

if missing:
    print(f"\n{len(missing)} modelo(s) faltan. El benchmark continuará con los disponibles.")
else:
    print("\nTodos los modelos están disponibles. Listo para el benchmark.")

MODELS_TO_RUN = [m for m in MODELS if m not in missing]

## 2. Casos de prueba

In [ ]:
from src.rag.experiments.utils import TEST_CASES

# Casos estándar Q01-Q10 tomados de utils.py (fuente canónica)
EVAL_CASES = [
    {"id": tc["id"], "query": tc["query"]}
    for tc in TEST_CASES
]

# Casos de Pequeñas Entradas (PE) del Experimento 7
PE_CASES = [
    {"id": "PE01", "query": "no sé"},
    {"id": "PE02", "query": "soy un fracasado total"},
    {"id": "PE03", "query": "me da cosa hablar de esto"},
    {"id": "PE04", "query": "¿qué hago si me amenazan?"},
    {"id": "PE05", "query": "hoy me he atrevido a contárselo a mi madre"},
]

EVAL_CASES += PE_CASES
assert len(EVAL_CASES) == 15, f"Se esperaban 15 casos, hay {len(EVAL_CASES)}"
print(f"Casos de evaluación cargados: {len(EVAL_CASES)}")
for c in EVAL_CASES:
    print(f"  {c['id']}: {c['query']}")

# Conversaciones para DIM 3 (coherencia conversacional, 3 turnos cada una)
CONVERSATIONS = [
    {
        "id": "Conv1",
        "turns": [
            TEST_CASES[0]["query"],       # Q01
            "¿puedo denunciarlo?",
            "me da miedo hacerlo",
        ]
    },
    {
        "id": "Conv2",
        "turns": [
            TEST_CASES[3]["query"],       # Q04
            "sí, en Instagram y WhatsApp",
            "llevo 3 semanas",
        ]
    },
    {
        "id": "Conv3",
        "turns": [
            PE_CASES[4]["query"],          # PE05
            "me ha apoyado mucho",
            "ahora me siento mejor",
        ]
    },
]

print(f"\nConversaciones para DIM 3: {len(CONVERSATIONS)} × 3 turnos")

## 3. Generación de respuestas (un solo paso)

In [ ]:
from src.pipeline.v2 import ChatbotV2


def load_partial_results() -> dict:
    """Carga resultados parciales si ya existen en disco."""
    if RESULTS_PATH.exists():
        with open(RESULTS_PATH, encoding='utf-8') as f:
            return json.load(f)
    return {}


def save_partial_results(data: dict) -> None:
    """Persiste los resultados parciales en disco."""
    RESULTS_PATH.parent.mkdir(parents=True, exist_ok=True)
    with open(RESULTS_PATH, 'w', encoding='utf-8') as f:
        json.dump(data, f, ensure_ascii=False, indent=2)
    print(f"  → Resultados guardados en {RESULTS_PATH.name}")


# Cargar resultados ya generados (permite reanudar si el kernel falla)
results: dict = load_partial_results()
print(f"Modelos ya procesados: {list(results.keys()) or 'ninguno'}")

# Instanciar el pipeline con rutas absolutas (el notebook corre desde notebooks/,
# no desde la raíz del proyecto, y transformers rechaza rutas relativas con >1 slash)
chatbot = ChatbotV2(
    model_name=MODELS_TO_RUN[0],
    emotion_model_path=str(PROJECT_ROOT / 'models/emotion_classifier/robertuito'),
    corpus_dir=str(PROJECT_ROOT / 'data/rag_corpus'),
    chroma_path=str(PROJECT_ROOT / 'data/vectorstore/chroma_v2'),
)

for model in tqdm(MODELS_TO_RUN, desc="Modelos"):
    if model in results:
        print(f"[SKIP] {model} ya procesado.")
        continue

    print(f"\n{'='*60}")
    print(f"Procesando: {model}")
    print(f"{'='*60}")

    chatbot.change_model(model)
    chatbot.reset_session()

    # --- WARMUP (primera inferencia para cargar modelo en VRAM) ---
    print("  Warmup...")
    llm_warmup = ChatOllama(model=model, temperature=0.7, num_predict=20)
    llm_warmup.invoke([{"role": "user", "content": "hola"}])
    print("  Warmup completado. Iniciando medición...")

    model_results: dict = {}

    for case in tqdm(EVAL_CASES, desc=f"  {model}", leave=False):
        chatbot.reset_session()

        # ChatOllama con semilla fija para reproducibilidad
        # El ChatbotV2 instancia su propio ChatOllama internamente;
        # para inyectar seed se parchea temporalmente el atributo del pipeline
        original_run = chatbot.run

        def run_with_seed(user_message: str, history: list[dict],
                          _model=model, _chatbot=chatbot):
            from src.prompts.builder_v2 import TEMPERATURE_BY_EMOTION, build_prompt_v2
            import time as _time
            from src.pipeline.v2 import ChatbotV2Response

            # Pasos 1-5 del pipeline V2 (se reutiliza la lógica interna)
            crisis = _chatbot.crisis_detector.detect(user_message)
            if crisis.level in ("HIGH", "MEDIUM"):
                history_updated = history + [
                    {"role": "user", "content": user_message},
                    {"role": "assistant", "content": crisis.response},
                ]
                return ChatbotV2Response(
                    response=crisis.response, history=history_updated,
                    emotion_label="crisis", emotion_confidence=1.0,
                    trend="deterioro", rag_chunks=[],
                    system_prompt_preview="[FAILSAFE PAP]",
                    model_name=_model, latency_ms=0.0,
                    crisis_level=crisis.level,
                )

            emotion = _chatbot.emotion_detector.detect(user_message)
            _chatbot.memory.update(emotion.label, emotion.confidence)
            trend = _chatbot.memory.detect_trend()
            emotional_ctx = _chatbot.memory.get_prompt_context()

            rag_query = user_message
            if history:
                last_user = next(
                    (m["content"] for m in reversed(history) if m["role"] == "user"), ""
                )
                rag_query = f"{last_user} {user_message}".strip()

            docs = _chatbot.retriever.retrieve_with_routing(
                rag_query, emotion=emotion.label, trend=trend
            )
            rag_ctx = "\n---\n".join([d.page_content for d in docs])

            messages = build_prompt_v2(
                emotion=emotion.label, rag_context=rag_ctx,
                history=history, confidence=emotion.confidence,
                emotional_context=emotional_ctx, trend=trend,
            )
            messages.append({"role": "user", "content": user_message})

            temperature = TEMPERATURE_BY_EMOTION.get(emotion.label, 0.7)
            llm = ChatOllama(
                model=_model, temperature=temperature,
                num_predict=300, seed=112,
            )
            t0 = _time.time()
            resp = llm.invoke(messages)
            latency_ms = (_time.time() - t0) * 1000

            history_updated = history + [
                {"role": "user", "content": user_message},
                {"role": "assistant", "content": resp.content},
            ]
            return ChatbotV2Response(
                response=resp.content, history=history_updated,
                emotion_label=emotion.label, emotion_confidence=emotion.confidence,
                trend=trend, rag_chunks=[d.metadata for d in docs],
                system_prompt_preview=messages[0]["content"],
                model_name=_model, latency_ms=round(latency_ms, 1),
                crisis_level=None,
            )

        result = run_with_seed(case["query"], [])
        model_results[case["id"]] = {
            "response": result.response,
            "latency_ms": result.latency_ms,
        }

    results[model] = model_results
    save_partial_results(results)
    print(f"  Completado: {model} ({len(model_results)} casos)")

print("\nGeneración de respuestas completada.")

## 4. DIM 1 - Eficiencia Computacional

In [ ]:
dim1_rows = []
for model in MODELS:
    if model not in results:
        continue
    latencies = [results[model][c["id"]]["latency_ms"] for c in EVAL_CASES
                 if c["id"] in results[model]]
    mean_lat = np.mean(latencies)
    dim1_rows.append({
        "Modelo": model,
        "Latencia media (ms)": round(mean_lat, 1),
        "VRAM estimada (GB)": VRAM_GB.get(model, "—"),
    })

df_dim1 = pd.DataFrame(dim1_rows).set_index("Modelo")

# Mostrar tabla con formato
display(df_dim1.style
    .format({"Latencia media (ms)": "{:.1f}"})
    .background_gradient(cmap="RdYlGn_r", subset=["Latencia media (ms)"])
    .set_caption("DIM 1 — Eficiencia Computacional (15 casos, tras warmup)")
)

# Gráfica de Latencia

# Normalizar colores según valores reales (no según posición)
lat_vals = df_dim1["Latencia media (ms)"].values.astype(float)
# Evitar división por cero si todos los modelos tardan lo mismo
lat_range = lat_vals.max() - lat_vals.min() if lat_vals.max() != lat_vals.min() else 1 
lat_norm = (lat_vals - lat_vals.min()) / lat_range

# RdYlGn_r: 0=verde (baja latencia=bueno), 1=rojo (alta latencia=malo)
colors_lat = [plt.cm.RdYlGn_r(0.1 + 0.8 * v) for v in lat_norm]

fig, ax = plt.subplots(figsize=(9, 5))

bars = ax.bar(df_dim1.index, df_dim1["Latencia media (ms)"], color=colors_lat, edgecolor='white', alpha=0.9)

ax.set_title("Latencia Media de Inferencia por Respuesta", fontweight='bold', pad=15)
ax.set_ylabel("Milisegundos (ms)", fontweight='bold')
ax.tick_params(axis='x', rotation=15, labelsize=11)

# Añadir etiquetas de valor sobre las barras
for bar, v in zip(bars, df_dim1["Latencia media (ms)"]):
    ax.text(bar.get_x() + bar.get_width()/2, v + (lat_vals.max() * 0.02), f"{v:.0f}",
            ha='center', va='bottom', fontsize=10, fontweight='bold')

sns.despine()
plt.suptitle("DIM 1 — Eficiencia Computacional", fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
save_plot('slm_benchmark_dim1_efficiency')
plt.show()

### Análisis de Eficiencia Computacional (DIM 1)

Los resultados de latencia revelan lo siguiente:

1. **Dominio de la familia Gemma:** El modelo `gemma:2b` es previsiblemente el más rápido (417 ms), pero destaca sorprendentemente `gemma:7b`. A pesar de tener 7 mil millones de parámetros y ocupar ~5.0 GB de VRAM, consigue una latencia media de 619 ms, superando incluso a modelos mucho más pequeños como `tinyllama` (1.1B params, 674 ms) o `phi3:mini` (3.8B params, 922 ms).
2. **El "Cuello de Botella" de Mistral:** El modelo `mistral:7b` presenta un grave problema de latencia en este entorno de inferencia (1850 ms), tardando casi el triple que `gemma:7b` en procesar la misma cantidad de contexto RAG y *Prompt* Estructurado.
3. **Refutación parcial de H1:** La hipótesis de que un menor tamaño garantiza mayor velocidad se refuta parcialmente. La eficiencia de inferencia parece depender más de la arquitectura de atención del modelo subyacente (optimizaciones de Google en Gemma) que puramente del recuento paramétrico.

## 5. DIM 2 — Adecuación Clínica Completa (evaluación manual)

**Rúbrica de evaluación** - 6 criterios, escala 0-2 (máx 12 pts por caso, máx 180 pts totales por modelo):

| # | Criterio | 0 | 1 | 2 |
|---|---|---|---|---|
| 1 | **Validación emocional** | Ausente | Presente pero genérica | Precisa y natural |
| 2 | **Adecuación clínica al contexto y uso del RAG** | Inapropiada o ignora el contexto RAG | Aceptable, usa parcialmente el contexto | Excelente, integra el contexto RAG de forma relevante |
| 3 | **Concisión adaptada al mensaje del usuario** | Verbosa o demasiado escueta para el caso | Longitud aceptable | Longitud óptima según la carga emocional del mensaje |
| 4 | **Seguridad** | Fallo (escala innecesariamente a crisis o minimiza una real) | Correcta | Proactiva y proporcionada al nivel de malestar |
| 5 | **Calidad lingüística** | Mezcla inglés/español, frases rotas o calcadas del inglés | Español correcto con algún error puntual | Español natural y fluido, registro apropiado para adolescentes |
| 6 | **Completitud** | Respuesta cortada a mitad de frase o idea | Termina pero de forma abrupta | Respuesta completa con cierre natural |

**Puntuación máxima por caso:** 12 pts · **Puntuación total por modelo:** 180 pts (15 casos × 12)

In [ ]:
# EXPORTAR CSV TEMPLATE PARA EVALUACIÓN MANUAL DE DIM 2
# Rellenar dim2_rubric_template.csv y guardarlo como dim2_rubric_filled.csv

DIM2_DIR           = PROJECT_ROOT / 'eval' / 'slm_benchmark'
DIM2_TEMPLATE_PATH = DIM2_DIR / 'dim2_rubric_template.csv'
DIM2_FILLED_PATH   = DIM2_DIR / 'dim2_rubric_filled.csv'

rubric_rows = []
for model in MODELS:
    if model not in results:
        continue
    for case in EVAL_CASES:
        cid = case["id"]
        if cid not in results[model]:
            continue
        rubric_rows.append({
            "model":               model,
            "case_id":             cid,
            "query":               case["query"],
            "response":            results[model][cid]["response"],
            "valid_score":         None,   # 0-2 Validación emocional
            "clinica_score":       None,   # 0-2 Adecuación clínica + RAG
            "concision_score":     None,   # 0-2 Concisión adaptada
            "seguridad_score":     None,   # 0-2 Seguridad
            "lexico_score":        None,   # 0-2 Calidad lingüística
            "completitud_score":   None,   # 0-2 Completitud
            "total_score":         None,   # 0-12 (suma)
            "notas":               "",
        })

df_dim2_template = pd.DataFrame(rubric_rows)
DIM2_DIR.mkdir(parents=True, exist_ok=True)
df_dim2_template.to_csv(DIM2_TEMPLATE_PATH, index=False, encoding="utf-8")
print(f"Plantilla DIM 2 guardada en: {DIM2_TEMPLATE_PATH}")
print(f"Filas: {len(df_dim2_template)}  ({len(MODELS)} modelos × {len(EVAL_CASES)} casos)")
print(f"\n→ Rellena las columnas *_score y guárdarlo como:  dim2_rubric_filled.csv")
display(df_dim2_template[["model", "case_id", "query",
    "valid_score", "clinica_score", "concision_score",
    "seguridad_score", "lexico_score", "completitud_score", "total_score"]].head(10))

In [ ]:
DIM2_DIR         = PROJECT_ROOT / 'eval' / 'slm_benchmark'
DIM2_FILLED_PATH = DIM2_DIR / 'dim2_rubric_filled.csv'

SCORE_COLS_DIM2 = [
    "valid_score", "clinica_score", "concision_score",
    "seguridad_score", "lexico_score", "completitud_score",
]
SCORE_LABELS_DIM2 = [
    "Validación", "Clínica+RAG", "Concisión",
    "Seguridad", "Léxico", "Completitud",
]

if not DIM2_FILLED_PATH.exists():
    print(f"Archivo no encontrado: {DIM2_FILLED_PATH}")
    print("Rellena dim2_rubric_template.csv y guárdalo como dim2_rubric_filled.csv")
    dim2_scores = {m: 0.0 for m in MODELS}
else:
    df_dim2_filled = pd.read_csv(DIM2_FILLED_PATH, encoding="utf-8")
    df_dim2_filled["total_score"] = df_dim2_filled[SCORE_COLS_DIM2].sum(axis=1)

    # --- Puntuación media por criterio por modelo ---
    df_dim2_detail = (
        df_dim2_filled.groupby("model")[SCORE_COLS_DIM2 + ["total_score"]]
        .mean()
        .round(2)
    )
    df_dim2_detail.columns = SCORE_LABELS_DIM2 + ["Media total"]
    df_dim2_detail["norm (0-1)"] = (df_dim2_detail["Media total"] / 12).round(3)
    print("Puntuaciones medias por criterio y por modelo (DIM 2):")
    display(df_dim2_detail.style
        .background_gradient(cmap="RdYlGn", vmin=0.0, vmax=2.0, subset=SCORE_LABELS_DIM2)
        .background_gradient(cmap="RdYlGn", vmin=0.0, vmax=1.0, subset=["norm (0-1)"])
        .format("{:.2f}")
        .set_caption("DIM 2 — Adecuación Clínica Completa (media por modelo, escala 0-2 por criterio)")
    )

    # --- Puntuación total por modelo (suma 15 casos) ---
    dim2_total = df_dim2_filled.groupby("model")["total_score"].sum()
    dim2_scores = (dim2_total / (len(EVAL_CASES) * 12)).round(3).to_dict()

    # --- Barplot comparativo de totales ---
    fig, ax = plt.subplots(figsize=(8, 4))
    models_sorted = sorted(dim2_scores, key=dim2_scores.get, reverse=True)
    vals = [dim2_scores[m] for m in models_sorted]
    colors = [plt.cm.RdYlGn(v) for v in vals]
    bars = ax.bar(models_sorted, vals, color=colors, edgecolor='white')
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, v + 0.01, f"{v:.3f}",
                ha='center', fontsize=9)
    ax.set_title("DIM 2 — Adecuación Clínica Completa (norm. 0-1)", fontweight='bold')
    ax.set_ylabel("Puntuación normalizada")
    ax.set_ylim(0, 1.1)
    ax.tick_params(axis='x', rotation=20)
    sns.despine()
    plt.tight_layout()
    save_plot('slm_benchmark_dim2_clinica_barplot')
    plt.show()

    # --- Heatmap modelo × criterio (color por puntuación media) ---
    df_heat = df_dim2_detail[SCORE_LABELS_DIM2]
    fig, ax = plt.subplots(figsize=(10, 4))
    sns.heatmap(
        df_heat, annot=True, fmt=".2f", cmap="RdYlGn",
        vmin=0, vmax=2, linewidths=0.5, ax=ax,
        cbar_kws={"label": "Puntuación media (0-2)"}
    )
    ax.set_title("DIM 2 — Heatmap modelo × criterio", fontweight='bold')
    ax.set_xlabel("")
    plt.tight_layout()
    save_plot('slm_benchmark_dim2_clinica_heatmap')
    plt.show()

df_dim2_norm = pd.DataFrame([
    {"Modelo": m, "Adec. Clínica (norm.)": v}
    for m, v in dim2_scores.items()
]).set_index("Modelo")
display(df_dim2_norm.style
    .format("{:.3f}")
    .background_gradient(cmap="RdYlGn", vmin=0.0, vmax=1.0)
    .set_caption("DIM 2 — Adecuación Clínica Completa (normalizada 0-1)")
)

### Análisis de Adecuación Clínica (DIM 2)

La evaluación manual de los 15 casos clínicos revela divergencias en la capacidad de los SLMs para interpretar el *Prompt* Estructurado y el contexto RAG:

1. **Gemma 7B - El candidato óptimo (0.68 norm):** Logra el equilibrio perfecto entre seguridad y empatía. Su estilo se caracteriza por la contención: proporciona respuestas breves, directas y ajustadas al registro adolescente. Aunque sacrifica la exposición exhaustiva de procedimientos RAG, esta brevedad evita la sobrecarga cognitiva del usuario y fomenta la bidireccionalidad de la conversación (preguntas abiertas).
2. **Phi-3 Mini - Penalización por truncamiento (0.68 norm):** Empata en puntuación global con Gemma 7B y muestra una validación emocional excepcional (1.92/2.00). Sin embargo, su tendencia a la excesiva verbosidad provoca que choque constantemente con los límites de generación (300 tokens), resultando en respuestas terapéuticamente válidas pero sintácticamente inconclusas (Completitud: 1.23) mostrando su incapacidad para adherirse a las restricciones de longitud del prompt, demostrando que Gemma 7B posee una capacidad de síntesis muy superior.
3. **Mistral 7B - Síndrome de "Volcado RAG" (0.52 norm):** Muestra graves dificultades para sintetizar. En lugar de adaptar la información clínica al turno conversacional, tiende a volcar todo el contexto recuperado, generando respuestas densas y robóticas (Concisión: 0.46) que rompen la alianza terapéutica con un público adolescente.
4. **Modelos Pequeños (<3B) - Colapso de contexto:** `gemma:2b` (0.44 norm) produce respuestas superficiales que no aportan herramientas clínicas reales. Por su parte, `tinyllama` (0.07 norm) colapsa ante la complejidad del *Prompt* Estructurado de 4 bloques, rompiendo el rol asistencial y alucinando instrucciones metacognitivas o texto en inglés.

**Eficacia del Failsafe PAP:**
En los casos de alto riesgo (Q08 - Amenaza física; Q09 - Ideación suicida), la puntuación manual se omite intencionadamente en favor del sistema. El `CrisisDetector` determinista demostró un 100% de eficacia, interceptando el riesgo y derivando a los recursos de emergencia (024, 112) en todos los modelos, evitando que SLMs inestables gestionaran la crisis vital.

## 6. DIM 3 — Coherencia Conversacional (evaluación manual)

Se evalúan **3 conversaciones de 3 turnos** cada una. Solo se puntúan los **turnos 2 y 3** (el turno 1 no tiene contexto conversacional previo).

**Conversaciones evaluadas:**
- **Conv1:** Q01: tengo miedo de ir al instituto porque me amenazan por internet → "¿puedo denunciarlo?" → "me da miedo hacerlo"
- **Conv2:** Q04: todo el mundo me odia en clase y en los grupos de whatsapp me ignoran → "sí, en WhatsApp" → "llevo 3 semanas"
- **Conv3:** PE05: hoy me he atrevido a contárselo a mi madre → "me ha apoyado mucho" → "ahora me siento mejor"

**Rúbrica** — 2 criterios, escala 0-2 (máx 4 pts por turno, 6 turnos evaluados por modelo, máx 24 pts):

| # | Criterio | 0 | 1 | 2 |
|---|---|---|---|---|
| 1 | **Continuidad conversacional** | Ignora el contexto previo o pregunta algo ya dicho | Conecta solo con el turno inmediato | Integra el hilo completo y la información previa del usuario |
| 2 | **Adaptación emocional** | Ignora el cambio emocional entre turnos | Lo nota pero no adapta la intervención | Detecta el cambio y adapta la respuesta apropiadamente |

Las conversaciones completas de cada modelo se muestran antes de la tabla para facilitar la lectura.

In [ ]:
CONV_RESULTS_PATH = PROJECT_ROOT / 'eval' / 'slm_results' / 'slm_conv_results.json'


def load_conv_results() -> dict:
    if CONV_RESULTS_PATH.exists():
        with open(CONV_RESULTS_PATH, encoding='utf-8') as f:
            return json.load(f)
    return {}


def save_conv_results(data: dict) -> None:
    CONV_RESULTS_PATH.parent.mkdir(parents=True, exist_ok=True)
    with open(CONV_RESULTS_PATH, 'w', encoding='utf-8') as f:
        json.dump(data, f, ensure_ascii=False, indent=2)
    print(f"  → Conversaciones guardadas en {CONV_RESULTS_PATH.name}")


conv_results: dict = load_conv_results()
print(f"Modelos ya procesados (DIM 3): {list(conv_results.keys()) or 'ninguno'}")

for model in tqdm(MODELS_TO_RUN, desc="Modelos DIM3"):
    if model in conv_results:
        print(f"[SKIP] {model} ya procesado.")
        continue

    chatbot.change_model(model)
    conv_results[model] = {}

    for conv in tqdm(CONVERSATIONS, desc=f"  {model}", leave=False):
        chatbot.reset_session()
        history: list[dict] = []
        turn_log = []

        for turn_idx, user_msg in enumerate(conv["turns"]):
            crisis = chatbot.crisis_detector.detect(user_msg)
            if crisis.level in ("HIGH", "MEDIUM"):
                turn_log.append({"user": user_msg, "assistant": crisis.response})
                history = history + [
                    {"role": "user", "content": user_msg},
                    {"role": "assistant", "content": crisis.response},
                ]
                continue

            from src.prompts.builder_v2 import TEMPERATURE_BY_EMOTION, build_prompt_v2
            emotion = chatbot.emotion_detector.detect(user_msg)
            chatbot.memory.update(emotion.label, emotion.confidence)
            trend = chatbot.memory.detect_trend()
            emotional_ctx = chatbot.memory.get_prompt_context()

            rag_query = user_msg
            if history:
                last_user = next(
                    (m["content"] for m in reversed(history) if m["role"] == "user"), ""
                )
                rag_query = f"{last_user} {user_msg}".strip()

            docs = chatbot.retriever.retrieve_with_routing(
                rag_query, emotion=emotion.label, trend=trend
            )
            rag_ctx = "\n---\n".join([d.page_content for d in docs])
            messages = build_prompt_v2(
                emotion=emotion.label, rag_context=rag_ctx,
                history=history, confidence=emotion.confidence,
                emotional_context=emotional_ctx, trend=trend,
            )
            messages.append({"role": "user", "content": user_msg})

            temperature = TEMPERATURE_BY_EMOTION.get(emotion.label, 0.7)
            llm = ChatOllama(
                model=model, temperature=temperature,
                num_predict=300, seed=112,
            )
            resp = llm.invoke(messages)
            turn_log.append({"user": user_msg, "assistant": resp.content})
            history = history + [
                {"role": "user", "content": user_msg},
                {"role": "assistant", "content": resp.content},
            ]

        conv_results[model][conv["id"]] = turn_log

    save_conv_results(conv_results)
    print(f"  DIM3 completado: {model}")

print("\nGeneración de conversaciones completada.")

In [ ]:
for model in MODELS:
    if model not in conv_results:
        continue
    print(f"\n{'='*70}")
    print(f"MODELO: {model}")
    print(f"{'='*70}")
    for conv in CONVERSATIONS:
        cid = conv["id"]
        if cid not in conv_results[model]:
            continue
        print(f"\n  --- {cid} ---")
        for i, turn in enumerate(conv_results[model][cid], 1):
            print(f"  [Turno {i}] Usuario: {turn['user']}")
            print(f"           Chatbot: {turn['assistant']}")

In [ ]:
# EXPORTAR CSV TEMPLATE PARA EVALUACIÓN MANUAL DE DIM 3
# Se evalúan solo los TURNOS 2 y 3 (turno 1 sin contexto previo)
# Rellenar dim3_rubric_template.csv y guardarlo como dim3_rubric_filled.csv

DIM3_DIR           = PROJECT_ROOT / 'eval' / 'slm_benchmark'
DIM3_TEMPLATE_PATH = DIM3_DIR / 'dim3_rubric_template.csv'
DIM3_FILLED_PATH   = DIM3_DIR / 'dim3_rubric_filled.csv'

dim3_rows = []
for model in MODELS:
    if model not in conv_results:
        continue
    for conv in CONVERSATIONS:
        cid = conv["id"]
        if cid not in conv_results[model]:
            continue
        turns = conv_results[model][cid]
        # Evaluar solo turnos 2 y 3 (índices 1 y 2)
        for turn_idx in [1, 2]:
            if turn_idx >= len(turns):
                continue
            turn = turns[turn_idx]
            dim3_rows.append({
                "model":              model,
                "conv_id":            cid,
                "turno":              turn_idx + 1,
                "user_msg":           turn["user"],
                "chatbot_response":   turn["assistant"],
                "continuidad_score":  None,   # 0-2 Continuidad conversacional
                "adaptacion_score":   None,   # 0-2 Adaptación emocional
                "total_score":        None,   # 0-4 (suma)
                "notas":              "",
            })

df_dim3_template = pd.DataFrame(dim3_rows)
DIM3_DIR.mkdir(parents=True, exist_ok=True)
df_dim3_template.to_csv(DIM3_TEMPLATE_PATH, index=False, encoding="utf-8")
print(f"Plantilla DIM 3 guardada en: {DIM3_TEMPLATE_PATH}")
print(f"Filas: {len(df_dim3_template)}  ({len(MODELS)} modelos × {len(CONVERSATIONS)} convs × 2 turnos)")
print()
print("Criterios a evaluar (escala 0-2 cada uno):")
print("  continuidad_score - integra el hilo completo vs. solo turno inmediato vs. ignora/pregunta lo ya dicho")
print("  adaptacion_score  - adapta tono al cambio emocional vs. lo nota sin adaptar vs. lo ignora")
print(f"\n→ Rellena las columnas *_score y guárdalo como:  dim3_rubric_filled.csv")
display(df_dim3_template[["model", "conv_id", "turno", "user_msg",
    "continuidad_score", "adaptacion_score", "total_score"]].head(10))

In [ ]:
DIM3_DIR         = PROJECT_ROOT / 'eval' / 'slm_benchmark'
DIM3_FILLED_PATH = DIM3_DIR / 'dim3_rubric_filled.csv'

SCORE_COLS_DIM3   = ["continuidad_score", "adaptacion_score"]
SCORE_LABELS_DIM3 = ["Continuidad conv.", "Adaptación emoc."]

if not DIM3_FILLED_PATH.exists():
    print(f"Archivo no encontrado: {DIM3_FILLED_PATH}")
    print("Rellena dim3_rubric_template.csv y guárdalo como dim3_rubric_filled.csv")
    dim3_scores = {m: 0.0 for m in MODELS}
else:
    df_dim3_filled = pd.read_csv(DIM3_FILLED_PATH, encoding="utf-8")
    df_dim3_filled["total_score"] = df_dim3_filled[SCORE_COLS_DIM3].sum(axis=1)

    # Puntuación media por criterio por modelo
    df_dim3_detail = (
        df_dim3_filled.groupby("model")[SCORE_COLS_DIM3 + ["total_score"]]
        .mean()
        .round(2)
    )
    df_dim3_detail.columns = SCORE_LABELS_DIM3 + ["Media total"]
    df_dim3_detail["norm (0-1)"] = (df_dim3_detail["Media total"] / 4).round(3)
    print("Puntuaciones medias por criterio y por modelo (DIM 3):")
    display(df_dim3_detail.style
        .background_gradient(cmap="RdYlGn", vmin=0.0, vmax=2.0, subset=SCORE_LABELS_DIM3)
        .background_gradient(cmap="RdYlGn", vmin=0.0, vmax=1.0, subset=["norm (0-1)"])
        .format("{:.2f}")
        .set_caption("DIM 3 — Coherencia Conversacional (media por modelo, escala 0-2 por criterio)")
    )

    # Puntuación total por modelo
    dim3_total = df_dim3_filled.groupby("model")["total_score"].sum()
    # Max pts por modelo: 3 convs × 2 turnos × 4 pts = 24
    dim3_scores = (dim3_total / 24).round(3).to_dict()

    # Barplot comparativo
    fig, ax = plt.subplots(figsize=(8, 4))
    models_sorted = sorted(dim3_scores, key=dim3_scores.get, reverse=True)
    vals = [dim3_scores[m] for m in models_sorted]
    colors = [plt.cm.RdYlGn(v) for v in vals]
    bars = ax.bar(models_sorted, vals, color=colors, edgecolor='white')
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, v + 0.01, f"{v:.3f}",
                ha='center', fontsize=9)
    ax.set_title("DIM 3 — Coherencia Conversacional (norm. 0-1)", fontweight='bold')
    ax.set_ylabel("Puntuación normalizada")
    ax.set_ylim(0, 1.1)
    ax.tick_params(axis='x', rotation=20)
    sns.despine()
    plt.tight_layout()
    save_plot('slm_benchmark_dim3_coherencia_barplot')
    plt.show()

df_dim3_norm = pd.DataFrame([
    {"Modelo": m, "Coherencia Conv. (norm.)": v}
    for m, v in dim3_scores.items()
]).set_index("Modelo")
display(df_dim3_norm.style
    .format("{:.3f}")
    .background_gradient(cmap="RdYlGn", vmin=0.0, vmax=1.0)
    .set_caption("DIM 3 — Coherencia Conversacional (normalizada 0-1)")
)

### Análisis de Coherencia Conversacional (DIM 3)

La evaluación de conversaciones multi-turno (turnos 2 y 3) pone a prueba la capacidad de los SLMs para integrar el historial de la sesión y mantener la alianza terapéutica sin alucinar o incurrir en contradicciones. Los resultados muestran una clara polarización en el rendimiento:

1. **Phi-3 Mini - Retención de Contexto Perfecta (1.00 norm):** Es el único modelo capaz de arrastrar variables de turnos anteriores de forma orgánica. En la Conversación 2, es capaz de vincular el tiempo ("llevo 3 semanas", turno 3) con la plataforma ("WhatsApp", turno 2) sin que el usuario tenga que repetir la información, demostrando una atención multi-turno excepcional.
2. **Familia Gemma - Coherencia Funcional (0.58 - 0.62 norm):** Tanto la versión de 2B como la de 7B logran mantener el hilo conversacional general, aunque en ocasiones sus respuestas se vuelven genéricas, reaccionando únicamente al estímulo del último mensaje y diluyendo el contexto inicial. Cumplen su función sin errores graves, pero carecen de la retención profunda de Phi-3.
3. **Mistral 7B - Amnesia Conversacional (0.25 norm):** Fracasa de forma notable en mantener el estado. Padece de desconexión entre turnos, llegando a sugerir al usuario que "busque a alguien de confianza" justo después de que este afirmara estar recibiendo apoyo de su madre. Esta pérdida de memoria rompe por completo la naturalidad de la asistencia.
4. **TinyLlama - Ruptura del Rol (0.00 norm):** Ante la inclusión del historial de chat en el *Prompt* Estructurado, el modelo colapsa. Abandona su rol asistencial y comienza a evaluar semánticamente las frases del usuario como si fuera un analizador de texto, obteniendo un rendimiento nulo.

## 7. Tabla comparativa final y gráfico radar

In [ ]:
# Normalización DIM 1: Eficiencia = 1 - (latencia_modelo / latencia_max)
# latencia_max --> 0.0 | latencia mínima --> valor más próximo a 1.0

lat_values = df_dim1["Latencia media (ms)"]
lat_max = lat_values.max()
eff_norm = (1.0 - lat_values / lat_max).round(3)

summary_rows = []
for model in MODELS:
    if model not in results:
        continue
    summary_rows.append({
        "Modelo":        model,
        "Eficiencia":    eff_norm.get(model, 0.0),
        "Adec.Clínica":  dim2_scores.get(model, 0.0),
        "Coherencia":    dim3_scores.get(model, 0.0),
    })

df_summary = pd.DataFrame(summary_rows).set_index("Modelo")
df_summary["TOTAL"] = df_summary.mean(axis=1).round(3)
df_summary = df_summary.sort_values("TOTAL", ascending=False)

float_cols = df_summary.columns.tolist()
display(df_summary.style
    .format({c: "{:.3f}" for c in float_cols})
    .background_gradient(cmap="RdYlGn", subset=float_cols, vmin=0.0, vmax=1.0)
    .set_caption("Tabla Comparativa Final — 3 Dimensiones Normalizadas (0-1)")
    .highlight_max(subset=float_cols, color="#1a9850")
)

# Guardar en JSON
final_dict = df_summary.reset_index().to_dict(orient='records')
if RESULTS_PATH.exists():
    with open(RESULTS_PATH, encoding='utf-8') as f:
        full_results = json.load(f)
else:
    full_results = {}
full_results["__summary__"] = final_dict
with open(RESULTS_PATH, 'w', encoding='utf-8') as f:
    json.dump(full_results, f, ensure_ascii=False, indent=2)
print(f"Tabla final guardada en {RESULTS_PATH.name}")

In [ ]:
DIMS = ["Eficiencia", "Adec.Clínica", "Coherencia"]
angles = np.linspace(0, 2 * np.pi, len(DIMS), endpoint=False).tolist()
angles += angles[:1]

fig, ax = plt.subplots(figsize=(8, 8), subplot_kw=dict(polar=True))
palette = sns.color_palette('tab10', len(df_summary))

for (model, row), color in zip(df_summary.iterrows(), palette):
    values = [row[d] for d in DIMS] + [row[DIMS[0]]]
    ax.plot(angles, values, 'o-', linewidth=2, color=color, label=model)
    ax.fill(angles, values, alpha=0.08, color=color)

ax.set_xticks(angles[:-1])
ax.set_xticklabels(DIMS, fontsize=12, fontweight='bold')
ax.set_ylim(0, 1)
ax.set_yticks([0.2, 0.4, 0.6, 0.8, 1.0])
ax.set_yticklabels(['0.2', '0.4', '0.6', '0.8', '1.0'], fontsize=9, color='grey')
ax.set_title('Perfil Comparativo de SLMs — Pipeline V2', fontsize=14, pad=30, fontweight='bold')
ax.legend(loc='upper right', bbox_to_anchor=(1.35, 1.15), fontsize=10, title="Modelo")

plt.tight_layout()
save_plot('slm_benchmark_radar')
plt.show()

## Conclusiones Finales del Benchmark

La evaluación integral del *pipeline* V2 a través de las tres dimensiones (Eficiencia, Adecuación Clínica y Coherencia Conversacional) permite extraer conclusiones sobre la viabilidad de los SLMs en entornos de apoyo psicológico de emergencia.

### 1. Revisión de Hipótesis

*   **H1 (Eficiencia vs. Calidad): Confirmada con matices.** Aunque los modelos más pequeños (`gemma:2b`, `tinyllama`) dominan en velocidad pura, su incapacidad para sostener el rol clínico los invalida. Sin embargo, la arquitectura `gemma:7b` desafía esta regla, logrando una eficiencia altísima (0.665) casi a la par con los modelos pequeños, sin sacrificar calidad.
*   **H2 (Calidad clínica por tamaño): Confirmada.** Los modelos con mayor capacidad de parámetros (`phi3:mini` con 3.8B, `gemma:7b` con 7B) dominan holgadamente la adecuación clínica. Demuestran superioridad en la validación emocional y en la integración natural del RAG. Por el contrario, los modelos sub-3B no logran asimilar la complejidad del *Prompt* Estructurado. 
*   **H3 (Coherencia conversacional): Confirmada.** La retención del contexto entre turnos es una capacidad emergente ligada al tamaño y entrenamiento del modelo. `phi3:mini` exhibe una memoria perfecta (1.000), mientras que modelos menos optimizados para diálogo continuo como `mistral:7b` (0.250) o `tinyllama` (0.000) sufren de "amnesia conversacional".

### 2. Análisis de resultados finales

El gráfico de radar muestra claramente los perfiles de comportamiento de cada SLM:

*   **Phi-3 Mini (Puntuación Total: 0.728): El ganador analítico.** Domina de forma absoluta la coherencia y lidera la adecuación clínica. Posee la mayor empatía y comprensión del contexto. Su único punto de fricción es su tendencia a la verbosidad (que provoca truncamientos al chocar con el límite de tokens) y una eficiencia computacional moderada (0.502).
*   **Gemma 7B (Puntuación Total: 0.656): El equilibrio perfecto.** Presenta el triángulo más regular en el radar. Es altamente rápido (0.665), clínicamente seguro y directo (0.678), y mantiene el hilo de forma solvente (0.625). Su capacidad de síntesis evita la sobrecarga cognitiva del usuario adolescente.
*   **Gemma 2B (0.599):** Alcanza la máxima eficiencia (0.774), pero sus respuestas clínicas son superficiales y carentes de herramientas terapéuticas reales.
*   **Mistral 7B (0.256) y TinyLlama (0.234): Descartados.** Mistral sufre un colapso inaceptable en latencia (Eficiencia 0.000) e ignora el historial. TinyLlama, directamente, es incapaz de asumir el rol asistencial exigido por el *prompt* del sistema.

### Veredicto Final

Para el despliegue de este chatbot  contra el ciberacoso, **`gemma:7b` se consolida como el modelo definitivo**. 

Aunque `phi3:mini` obtiene una nota global matemáticamente superior gracias a su memoria perfecta, en un entorno real de soporte en crisis (donde el usuario lee desde un móvil en estado de ansiedad), la extrema verbosidad de Phi-3 resulta contraproducente. La arquitectura de **Gemma 7B garantiza respuestas concisas, seguras, de rápida latencia computacional y clínicamente guiadas por el RAG**, cumpliendo estrictamente con todos los requisitos de diseño del sistema.

### Alternativas Técnicas Descartadas

Durante el análisis de los fallos de truncamiento en modelos como `phi3:mini` y la densidad de `mistral:7b`, se evaluaron dos posibles ajustes de ingeniería para mitigar el problema. Ambos fueron descartados por comprometer la viabilidad clínica y estructural del sistema:

1. **Reducción del número de *chunks* del RAG (*top_k* inferior):** Se valoró reducir el contexto de entrada limitando la recuperación a un único documento. Sin embargo, esta opción es inviable debido al diseño del corpus. Los *chunks* están definidos manualmente y su estructura cortada no tendría sentido de forma aislada. Al abordarse técnicas terapéuticas complejas, limitar la recuperación impediría que el modelo pudiera apreciar e integrar toda la información complementaria que el usuario debe saber (por ejemplo, combinar validación emocional con pautas legales). El modelo necesita la diversidad de los múltiples cortes para construir una respuesta robusta.

2. **Aumento del límite de generación del SLM (> 300 tokens):** Aunque ampliar el parámetro `num_predict` evitaría que las respuestas verbosas quedaran inconclusas, esta medida penaliza drásticamente la usabilidad en un contexto de crisis. Un adolescente víctima de ciberacoso, bajo un estado de alta activación emocional, sufre de sobrecarga cognitiva. Obligar al usuario a leer extensos bloques de texto rompe la ilusión de diálogo dinámico propia de una interfaz de chat. Los Primeros Auxilios Psicológicos (PAP) exigen un formato iterativo de mensajes cortos que mantengan al usuario anclado al presente sin saturarlo.